# 4. The Earth Similarity Index -- and a real bug in its history

The Schulze-Makuch et al. (2011) ESI, the equilibrium-temperature
substitution this project makes explicit rather than hiding, and a genuine
exponent bug an external audit caught in this exact codebase -- reproduced
here numerically so the failure mode is visible, not just described.


This notebook is part of the reproducibility set for **Finding Earth 2.0 in
Distant Worlds**. It reads the same committed data every other output in this
project reads (`results/`, `data/processed/`, `data/manifests/`) and calls
the same `earth2` functions the pipeline itself calls -- nothing here is a
simplified restatement computed a different way. Run `python -m earth2 all`
first if `results/` does not exist yet.

See `docs/METHODS.md` for the full equations and `docs/LIMITATIONS.md` for
this project's stated caveats.


In [1]:
import sys
sys.path.insert(0, "../src")

import numpy as np

from earth2.habitability import esi
from earth2.constants import SOLAR_SYSTEM_CONTROLS

print("ESI_WEIGHTS:", esi.ESI_WEIGHTS)
print("Earth's own equilibrium-temperature reference (K):", round(esi.T_EQ_EARTH_REFERENCE_K, 3))


ESI_WEIGHTS: {'radius': 0.57, 'density': 1.07, 'escape_velocity': 0.7, 'temperature': 5.58}
Earth's own equilibrium-temperature reference (K): 254.194


## Earth scores exactly 1.0, by construction

The reference values ARE Earth's own values, so every term's fractional
difference is zero and every term equals 1.0.


In [2]:
earth_esi = esi.esi_global(
    radius_earth=1.0,
    density_g_cm3=esi.ESI_REFERENCES["density"],
    escape_velocity_kms=esi.ESI_REFERENCES["escape_velocity"],
    equilibrium_temp_k=esi.T_EQ_EARTH_REFERENCE_K,
)
{k: round(float(v), 6) for k, v in earth_esi.items()}


{'esi_radius': 1.0,
 'esi_density': 1.0,
 'esi_escape_velocity': 1.0,
 'esi_temperature': 1.0,
 'esi_interior': 1.0,
 'esi_surface': 1.0,
 'esi_global': 1.0}

## Venus: near-perfect bulk similarity, radically different surface

Venus's radius, density and escape velocity are all close to Earth's, and
its equilibrium temperature (driven by its very high Bond albedo) is
actually *cooler* than Earth's -- the ESI, computed from data that exists
for real exoplanets, cannot see the 92-bar CO2 atmosphere that makes its
actual surface 737 K.


In [3]:
v = SOLAR_SYSTEM_CONTROLS["Venus"]
v_vesc = float(esi.escape_velocity_earth_units(v["pl_bmasse"], v["pl_rade"]))
venus_esi = esi.esi_global(v["pl_rade"], v["pl_dens"], v_vesc, v["pl_eqt"])
print("Venus ESI_global:", round(float(venus_esi["esi_global"]), 4))
print("Venus ESI_temperature alone:", round(float(venus_esi["esi_temperature"]), 4))
print("(Venus's real surface temperature is 737 K -- this metric cannot see that.)")


Venus ESI_global: 0.9199
Venus ESI_temperature alone: 0.8761
(Venus's real surface temperature is 737 K -- this metric cannot see that.)


## A real bug, reproduced numerically

An external repository audit of this project found that an earlier version
of `esi_global()` took **one extra square root** when combining the two
property tiers, compounding the exponent to `w_i/8` instead of the published
`w_i/4`. Both versions give Earth = 1.0 exactly (every term is 1, and
`1**anything == 1`) -- which is exactly why the bug was invisible to a test
that only checked Earth. It is visible the moment you check anything else.


In [4]:
def esi_global_buggy(radius, density, vesc, teq):
    '''The pre-fix version: one incorrect extra sqrt when forming each tier.'''
    n_tier = 2
    er = esi.esi_component(radius, esi.ESI_REFERENCES["radius"], esi.ESI_WEIGHTS["radius"], n_tier)
    ed = esi.esi_component(density, esi.ESI_REFERENCES["density"], esi.ESI_WEIGHTS["density"], n_tier)
    ev = esi.esi_component(vesc, esi.ESI_REFERENCES["escape_velocity"], esi.ESI_WEIGHTS["escape_velocity"], n_tier)
    et = esi.esi_component(teq, esi.ESI_REFERENCES["temperature"], esi.ESI_WEIGHTS["temperature"], n_tier)
    interior = np.sqrt(er * ed)   # <- the bug: er, ed already carry w/2; this adds an extra /2
    surface = np.sqrt(ev * et)
    return float(np.sqrt(interior * surface))

buggy = esi_global_buggy(v["pl_rade"], v["pl_dens"], v_vesc, v["pl_eqt"])
fixed = float(venus_esi["esi_global"])
print(f"Buggy  (w_i/8): Venus ESI = {buggy:.4f}")
print(f"Fixed  (w_i/4): Venus ESI = {fixed:.4f}")
print(f"The bug compresses every non-Earth score toward 1.0 -- Venus reads "
      f"{100*(buggy-fixed)/fixed:+.1f}% higher than it should.")


Buggy  (w_i/8): Venus ESI = 0.9591
Fixed  (w_i/4): Venus ESI = 0.9199
The bug compresses every non-Earth score toward 1.0 -- Venus reads +4.3% higher than it should.


## Independent cross-check against the flat four-variable formula

The published formula, written directly with no tier structure at all, must
agree with the (correct) tiered implementation to floating-point precision.
This is exactly the regression test added to `tests/test_habitability_esi.py`
after the fix.


In [5]:
def esi_flat(radius, density, vesc, teq):
    total = 1.0
    for value, key in ((radius, "radius"), (density, "density"),
                       (vesc, "escape_velocity"), (teq, "temperature")):
        ref = esi.ESI_REFERENCES[key]
        frac = abs((value - ref) / (value + ref))
        total *= (1.0 - frac) ** (esi.ESI_WEIGHTS[key] / 4.0)
    return total

flat = esi_flat(v["pl_rade"], v["pl_dens"], v_vesc, v["pl_eqt"])
print("Tiered implementation:", round(fixed, 9))
print("Flat four-variable formula:", round(flat, 9))
assert abs(fixed - flat) < 1e-9
print("Agree to 1e-9 -- the tier structure is a reporting convenience, not a different formula.")


Tiered implementation: 0.919903516
Flat four-variable formula: 0.919903516
Agree to 1e-9 -- the tier structure is a reporting convenience, not a different formula.
